# 📦 FaceKey Studio — Gen1 Cloud-to-Drive Migration & Archival
### Safely Transfer All Gen1 Data (~54.6 GB Unique Chunks) and Trained Model Checkpoints into 5 TB Google Drive

> [!IMPORTANT]
> **SAFETY PROTOCOL:**
> 1. **Zero Data Loss**: This notebook transfers all unique dataset chunks and trained model weights into `/content/drive/MyDrive/FaceKeyDataset/Gen1/`.
> 2. **Duplicate-Proof**: Automatically filters out all 16 redundant `*(1).tar.gz` files from Hugging Face, saving 14 GB of bandwidth.
> 3. **Colab VM Disk Protection**: Downloads 1 chunk at a time to `/tmp` scratch, commits directly to Drive, and immediately purges scratch memory so Colab local disk usage stays < 500 MB at all times.
> 4. **User Verification Gate**: Do NOT delete anything from Hugging Face until you manually inspect your Google Drive and confirm that all data and model checkpoints are safely stored!

## ⚡ STEP 1: MOUNT 5 TB GOOGLE DRIVE

In [ ]:
# Mount Google Drive to /content/drive
from google.colab import drive
import os

drive.mount('/content/drive')

GEN1_DIR = '/content/drive/MyDrive/FaceKeyDataset/Gen1'
os.makedirs(f'{GEN1_DIR}/chunks', exist_ok=True)
os.makedirs(f'{GEN1_DIR}/checkpoints', exist_ok=True)

print(f'[✓] Google Drive mounted successfully!')
print(f'[✓] Gen1 Archive directory ready: {GEN1_DIR}')

## ⚡ STEP 2: CLONE & PULL LATEST FACEKEY MIGRATION CODE

In [ ]:
# Pull latest code from GitHub
%cd /content
APP_DIR = '/content/FaceKeyAnimation'

if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub code...')
    !cd {APP_DIR} && git fetch --all && git reset --hard origin/main
else:
    print('[*] Cloning FaceKeyAnimation from GitHub...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}
!pip install -q -U huggingface_hub
print('[✓] Migration environment ready!')

## ⚡ STEP 3: RUN ZERO-DISK DUPLICATE-PROOF GEN1 MIGRATION ENGINE

In [ ]:
# Run Gen1 migration engine
# - Adopts any existing chunks already on Drive (0 bandwidth)
# - Downloads all Gen1 model checkpoints into Gen1/checkpoints/
# - Downloads all missing unique chunks into Gen1/chunks/
# - Purges scratch memory after each chunk (Colab disk safe!)
%cd {APP_DIR}
!python scripts/migrate_gen1_hf_to_drive.py

## 🔍 STEP 4: VERIFY GEN1 DRIVE ARCHIVE & DISPLAY LEDGER

In [ ]:
# Inspect and print verification statistics
import glob, json, os

GEN1_DIR = '/content/drive/MyDrive/FaceKeyDataset/Gen1'
chunks = glob.glob(f'{GEN1_DIR}/chunks/*.tar.gz')
ckpts = glob.glob(f'{GEN1_DIR}/checkpoints/*')
ledger_file = f'{GEN1_DIR}/GEN1_DATASET_LEDGER.json'

total_chunk_bytes = sum(os.path.getsize(p) for p in chunks)
total_ckpt_bytes = sum(os.path.getsize(p) for p in ckpts)

print('=' * 75)
print(' 📊 GEN1 GOOGLE DRIVE ARCHIVE VERIFICATION REPORT')
print('=' * 75)
print(f' • Total Gen1 Chunks Secured      : {len(chunks)} chunks ({total_chunk_bytes / (1024**3):.2f} GB)')
print(f' • Total Gen1 Checkpoints Secured : {len(ckpts)} files ({total_ckpt_bytes / (1024**2):.1f} MB)')
for c in ckpts:
    print(f'    - {os.path.basename(c)} ({os.path.getsize(c) / (1024**2):.1f} MB)')

if os.path.exists(ledger_file):
    with open(ledger_file) as f:
        ledger = json.load(f)
    print(f' • Archival Status                : {ledger.get("archive_status", "N/A")}')
    print(f' • Ledger Timestamp               : {ledger.get("last_updated", "N/A")}')
print('=' * 75)
print('\n👉 MANUAL VERIFICATION STEP:')
print('   Open Google Drive in your browser, navigate to "FaceKeyDataset/Gen1/",')
print('   and visually confirm the chunks and checkpoints folders before proceeding to Gen2.')